In [2]:
import re
from datetime import timedelta
from bisect import bisect_left

import re
from datetime import timedelta

# Updated helper function to convert VTT timestamp to seconds (including milliseconds)
def timestamp_to_seconds(ts):
    parts = re.split('[:.]', ts)
    if len(parts) == 4:
        h, m, s, ms = parts
    elif len(parts) == 3:
        h, m, s = parts
        ms = 0
    else:
        raise ValueError(f"Unexpected timestamp format: {ts}")
    return int(h) * 3600 + int(m) * 60 + int(s) + int(ms) / 1000.0



# Helper function to convert seconds to VTT timestamp
def seconds_to_timestamp(seconds):
    td = timedelta(seconds=seconds)
    total_seconds = int(td.total_seconds())
    hours = total_seconds // 3600
    minutes = (total_seconds % 3600) // 60
    secs = total_seconds % 60
    millis = int((td.total_seconds() - total_seconds) * 1000)
    return f"{hours:02}:{minutes:02}:{secs:02}.{millis:03}"

# Parse VTT file into list of (start, end, text)
def parse_vtt(filename):
    with open(filename, 'r', encoding='utf-8') as f:
        content = f.read()
    entries = []
    blocks = re.split(r'\n\n+', content)
    for block in blocks:
        lines = block.strip().split('\n')
        if len(lines) >= 2 and '-->' in lines[0]:
            times = lines[0].split('-->')
            start = timestamp_to_seconds(times[0].strip())
            end = timestamp_to_seconds(times[1].strip())
            text = '\n'.join(lines[1:])
            entries.append((start, end, text))
    return entries

# Build a timeline of merged subtitles
def merge_vtt_tracks(track1, track2, interval=1.0):
    # Determine global time range
    all_times = sorted(set([t for s, e, _ in track1 + track2 for t in (s, e)]))
    start_time = all_times[0]
    end_time = all_times[-1]

    # Create time segments
    timeline = []
    t = start_time
    while t < end_time:
        timeline.append((t, t + interval))
        t += interval

    # Function to collect texts overlapping a time segment
    def collect_texts(track, start, end):
        return '\n'.join([text for s, e, text in track if s < end and e > start])

    # Merge segments
    merged = []
    for start, end in timeline:
        text1 = collect_texts(track1, start, end)
        text2 = collect_texts(track2, start, end)
        if text1 or text2:
            combined_text = f"{text1}\n{text2}".strip()
            merged.append((start, end, combined_text))
    return merged

# Write merged subtitles to VTT file
def write_vtt(filename, merged_entries):
    with open(filename, 'w', encoding='utf-8') as f:
        f.write("WEBVTT\n\n")
        for i, (start, end, text) in enumerate(merged_entries):
            f.write(f"{seconds_to_timestamp(start)} --> {seconds_to_timestamp(end)}\n{text}\n\n")

# Example usage:
# track1 = parse_vtt("track1.vtt")
# track2 = parse_vtt("track2.vtt")
# merged = merge_vtt_tracks(track1, track2, interval=1.0)
# write_vtt("merged.vtt", merged)


In [5]:
#base_dir='/Users/kesiyun/Desktop/00workspace/videos/'
vtt_output_dir='/Users/kesiyun/Desktop/00workspace/VTTs/0/'
base_dir='/Users/kesiyun/Desktop/00workspace/VTTs/0bkup/' #/Users/kesiyun/Desktop/00workspace/VTTs/0bkup/17I.test.VTT

test_set_videos=['4E', '12B', '17I', '4C', '16A', '4A']
#videoname='17I' #333 '4E' '12B'   #22 '17I' '4C' # 1 16A
for videoname in test_set_videos:
    #visualized_str_pred=visualize_c_tensor_pred(videoname)
    #visualized_str_pred = '_________________________________________________________________________________II_II___________________________________________________________________________________________________________________________________I_____________________________________________________________________________________________________________________________________________________________________________________________________________I__________________________________________________'
    #visualized_str_pred='qwert12345yuiop67890qwert12345yuiop67890qwert12345yuiop67890'
    track1f=videoname+'.test.VTT'
    track2f=videoname+'.out.VTT'
    mergedf=videoname+'.VTT'
    track1dir=base_dir+track1f
    track2dir=vtt_output_dir+track2f
    mergeddir=vtt_output_dir+mergedf
    

    track1 = parse_vtt(track1dir)  # First subtitle file
    track2 = parse_vtt(track2dir)  # Second subtitle file
    merged = merge_vtt_tracks(track1, track2, interval=1.0)  # Merge with 1s intervals
    write_vtt(mergeddir, merged)  # Output to new VTT file